# Predictive Process Monitoring — Suffix Prediction Benchmark

This notebook provides a **single entry point** to reproduce all experiments from the paper:

> *SuTraN: an Encoder-Decoder Transformer for Full-Context-Aware Suffix Prediction of Business Processes*  
> ICPM 2024

It also includes the **BEST** baseline introduced in:

> *BEST: Bilaterally Expanding Subtrace Tree for Event Sequence Prediction*  
> Rauch et al., BPM 2025

---

## What this notebook does

1. **Creates pre-processed tensor datasets** from the raw event logs (BPIC17, BPIC17-DR, BPIC19).
2. **Trains and evaluates every model** on each event log and saves results to disk.

Each model section includes a short description, the paper it is based on, and a code cell that runs `train_eval()` for every log.

---

## Models covered

| # | Model | Type | Paper |
|---|-------|------|-------|
| 1 | **SuTraN (DA)** | Encoder-Decoder Transformer | SuTraN paper (ICPM 2024) |
| 2 | **SuTraN (NDA)** | Encoder-Decoder Transformer (control-flow only) | SuTraN paper (ICPM 2024) |
| 3 | **CRTP-LSTM (DA)** | Multi-task LSTM | Camargo et al. (2019) |
| 4 | **CRTP-LSTM (NDA)** | Multi-task LSTM (control-flow only) | Camargo et al. (2019) |
| 5 | **ED-LSTM** | Encoder-Decoder LSTM (seq2seq) | Tax et al. (2017) / Pfeiffer et al. (2021) |
| 6 | **SEP-LSTM** | One-step-ahead LSTM with iterative feedback | Taymouri et al. (2021) |
| 7 | **BEST** | N-gram pattern tree (no neural network) | Rauch et al., BPM 2025 |

**DA** = Data-Aware (uses all event / case attributes)  
**NDA** = Non-Data-Aware (uses only activity labels and timestamp proxies)

---

## Event logs

| Log | Cases | Mean length | Raw file |
|-----|-------|-------------|----------|
| BPIC17 | 30,078 | 36.90 | `bpic17_with_loops.csv` |
| BPIC17-DR | 30,078 | 23.42 | `BPIC17_no_loop.csv` |
| BPIC19 | 181,395 | 5.44 | `BPIC19.csv` |

Place the CSV files in the repository root before running the dataset creation cells.

In [1]:
# ── Shared configuration ────────────────────────────────────────────────────
# Log names must match the log_name used in log_to_tensors() (set inside each
# create_*_data.py script).  tss_index is the zero-based position of the
# 'time since start' column inside the numerical prefix feature tensor;
# it equals the number of numerical event + case features for that log.

LOGS = [
    #{"log_name": "BPIC_17",    "tss_index": 5},  # 4 num. event fts + 1 num. case ft
    #{"log_name": "BPIC_17_DR", "tss_index": 5},  # same numerical features, no loops
    #{"log_name": "BPIC_19",    "tss_index": 1},  # 1 num. event ft, no num. case fts
    {"log_name": "Sepsis",    "tss_index": 4},  
]

print("Logs configured:", [l['log_name'] for l in LOGS])

Logs configured: ['Sepsis']


---
# Part 1 — Creating Datasets

Each script reads a raw CSV event log, applies log-specific preprocessing (timestamp parsing, invalid-case removal, feature engineering), then calls `log_to_tensors()` from `Preprocessing/from_log_to_tensors.py`.

The pipeline:
1. Filters cases by date range and maximum duration.
2. Maps categorical features to integers.
3. Standardises numerical features using training-set statistics.
4. Creates all prefix–suffix pairs (sliding window).
5. Splits into train / validation / test using an **out-of-time** split with data-leakage prevention.
6. Saves `train_tensordataset.pt`, `val_tensordataset.pt`, `test_tensordataset.pt` and metadata pickles inside a subfolder named after the log.

> **Note:** Run each cell only once. Re-running will overwrite the saved tensors.

## Dataset 1 — BPIC17

Loan-application process recorded at a Dutch financial institution (BPI Challenge 2017).  
Source: [doi.org/10.4121/uuid:5f3067df-f10b-45da-b98b-86ae4c7a310b](https://doi.org/10.4121/uuid:5f3067df-f10b-45da-b98b-86ae4c7a310b)

- **Cases:** 30,078 &emsp; **Events:** 1,109,665 &emsp; **Activities:** 25
- **Window size:** 88 &emsp; **End date filter:** 2017-01
- **Features:** 3 categorical case fts · 3 categorical event fts · 4 numerical event fts · 1 numerical case ft

**Raw file required:** `bpic17_with_loops.csv`

In [4]:
from create_BPIC17_OG_data import construct_BPIC17_datasets

construct_BPIC17_datasets()

Generating Dataframes...


/app/Preprocessing/create_benchmarks.py:85: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  case_stops_df['date'] = case_stops_df[timestamp].dt.to_period('M')
100%|█████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 50.57it/s]


Generating Tensors...
Computing train set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
____________________________
Computing validation set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
____________________________
Computing test set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...


## Dataset 2 — BPIC17-DR *(De-Looped)*

Derived from BPIC17 by removing repeated (*looping*) activities within the same case, resulting in simpler, shorter traces.

- **Cases:** 30,078 &emsp; **Events:** 704,202 &emsp; **Activities:** 25
- **Window size:** 46 &emsp; **End date filter:** 2017-01
- **Features:** 2 categorical case fts · 3 categorical event fts · 4 numerical event fts · 1 numerical case ft

**Raw file required:** `BPIC17_no_loop.csv`

In [2]:
from create_BPIC17_DR_data import construct_BPIC17_DR_datasets

construct_BPIC17_DR_datasets()

Generating Dataframes...


/app/Preprocessing/create_benchmarks.py:85: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  case_stops_df['date'] = case_stops_df[timestamp].dt.to_period('M')
100%|█████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 69.61it/s]


Generating Tensors...
Computing train set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
____________________________
Computing validation set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
____________________________
Computing test set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...


## Dataset 3 — BPIC19

Purchase-order handling process from a multinational company (BPI Challenge 2019).  
Source: [doi.org/10.4121/uuid:d06aff4b-79f0-45e6-8ec8-e19730c248f1](https://doi.org/10.4121/uuid:d06aff4b-79f0-45e6-8ec8-e19730c248f1)

- **Cases:** 181,395 &emsp; **Events:** 986,077 &emsp; **Activities:** 39
- **Window size:** 17 &emsp; **Date range:** 2018-01 → 2019-02
- **Features:** 11 categorical case fts · 1 categorical event ft · 1 numerical event ft

**Raw file required:** `BPIC19.csv`

In [2]:
from create_BPIC19_data import construct_BPIC19_datasets

construct_BPIC19_datasets()

Generating Dataframes...


/app/Preprocessing/create_benchmarks.py:58: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  case_starts_df['date'] = case_starts_df[timestamp].dt.to_period('M')
/app/Preprocessing/create_benchmarks.py:85: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  case_stops_df['date'] = case_stops_df[timestamp].dt.to_period('M')
/app/Preprocessing/create_benchmarks.py:115: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  case_starts_df['date'] = case_starts_df[timestamp].dt.to_period('M')
100%|███████████████████████████████████████████████████████████████| 13/13 [00:00<00:00, 55.41it/s]


Generating Tensors...
Computing train set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
____________________________
Computing validation set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
____________________________
Computing test set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...


# Any dataset

In [1]:
from create_general_data import construct_datasets

# ------------------------------------------------------------------ #
# Edit the variables below to match your event log.                   #
# ------------------------------------------------------------------ #

LOG_PATH   = 'Logs/Sepsis.xes.gz'   # or 'my_log.csv'
LOG_NAME   = 'Sepsis'

# Standard XES column names — change if your CSV uses different names.
CASE_ID    = 'case:concept:name'
ACT_LABEL  = 'concept:name'
TIMESTAMP  = 'time:timestamp'

# Set to None to auto-detect from column prefixes / dtypes, or provide
# explicit lists as in the log-specific create_*.py files.
CAT_CASEFTS  = None
NUM_CASEFTS  = None
CAT_EVENTFTS = None
NUM_EVENTFTS = None

# Filtering / splitting parameters — adjust to your log's date range
# and case-length distribution.
START_DATE        = None   # e.g. "2018-01"
START_BEFORE_DATE = None   # e.g. "2018-09"
END_DATE          = None   # e.g. "2019-02"
MAX_DAYS          = None   # e.g. 143.33
WINDOW_SIZE       = None   # None → auto (98.5th percentile)
TEST_LEN_SHARE    = 0.25
VAL_LEN_SHARE     = 0.20
MODE              = 'preferred' # preferred or workaround
OUTCOME           = None
PLOT              = True    # set to False to skip the split visualisation

construct_datasets(
    log_path=LOG_PATH,
    log_name=LOG_NAME,
    case_id=CASE_ID,
    act_label=ACT_LABEL,
    timestamp=TIMESTAMP,
    cat_casefts=CAT_CASEFTS,
    num_casefts=NUM_CASEFTS,
    cat_eventfts=CAT_EVENTFTS,
    num_eventfts=NUM_EVENTFTS,
    outcome=OUTCOME,
    start_date=START_DATE,
    start_before_date=START_BEFORE_DATE,
    end_date=END_DATE,
    max_days=MAX_DAYS,
    window_size=WINDOW_SIZE,
    test_len_share=TEST_LEN_SHARE,
    val_len_share=VAL_LEN_SHARE,
    mode=MODE,
    plot=PLOT,
)

parsing log, completed traces ::   0%|          | 0/1050 [00:00<?, ?it/s]

Split plot saved to 'results_per_log/Sepsis/Sepsis_preferred_split.png'
Auto-detected features:
  cat_casefts  : []
  num_casefts  : []
  cat_eventfts : ['InfectionSuspected', 'org:group', 'DiagnosticBlood', 'DisfuncOrg', 'SIRSCritTachypnea', 'Hypotensie', 'SIRSCritHeartRate', 'Infusion', 'DiagnosticArtAstrup', 'DiagnosticIC', 'DiagnosticSputum', 'DiagnosticLiquor', 'DiagnosticOther', 'SIRSCriteria2OrMore', 'DiagnosticXthorax', 'SIRSCritTemperature', 'DiagnosticUrinaryCulture', 'SIRSCritLeucos', 'Oligurie', 'DiagnosticLacticAcid', 'lifecycle:transition', 'Diagnose', 'Hypoxie', 'DiagnosticUrinarySediment', 'DiagnosticECG']
  num_eventfts : ['Age', 'Leucocytes', 'CRP', 'LacticAcid']
Auto-derived window_size (98.5th percentile): 41
Auto-derived max_days (maximum case duration): 422.32
Generating Dataframes...


100%|█████████████████████████████████████████████████████████| 26/26 [00:00<00:00, 620.64it/s]


Cases – train: 551  val: 138  test: 345
Pairs – train: 7367  val: 1782  test: 4837
Generating Tensors...
Computing train set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
____________________________
Computing validation set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
____________________________
Computing test set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
Tensors saved to 'results_per_log/Sepsis/'
tss_index = 4  (= 0 num_casefts + 4 num_eventfts)
Use this value for the tss_index parameter in TRAIN_EVAL_*.py scripts.


---
# Part 2 — Model Training & Evaluation

Each section below trains one model on every event log and automatically evaluates it on the held-out test set.  Results (DL similarity, TTNE MAE, RRT MAE, per-length dictionaries) are saved under `<log_name>/<model>_results/TEST_SET_RESULTS/`.

Training is GPU-accelerated when a CUDA device is available.  Models use early stopping based on combined validation ranking of DL similarity and RRT MAE.

---
## Model 1 — SuTraN (Data-Aware)

**Approach from paper:** *SuTraN: an Encoder-Decoder Transformer for Full-Context-Aware Suffix Prediction of Business Processes* (ICPM 2024)

SuTraN is an **encoder-decoder Transformer** designed specifically for multi-task suffix prediction in PPM.  Key properties:

- The **encoder** is a stack of self-attention layers that builds a full contextual representation of the observed prefix, using *all* available event and case attributes (Data-Aware).
- The **decoder** generates the complete activity and timestamp suffix in a **single forward pass** during training (teacher forcing), and autoregressively at inference.
- **Cross-attention** at every decoder step lets each predicted suffix token attend to the entire encoded prefix — giving the model its *full-context-aware* property.
- **Multi-task heads** simultaneously predict: activity labels, time-till-next-event (TTNE) at each suffix step, and a direct remaining runtime (RRT) estimate.
- Activity embeddings are **shared** between encoder and decoder.

Architecture defaults: `d_model=32`, 4 encoder + 4 decoder layers, 8 attention heads, `d_ff=128`, dropout 0.2, AdamW + exponential LR decay, up to 200 epochs with patience 24.

In [2]:
import TRAIN_EVAL_SUTRAN_DA as sutran_da

for cfg in LOGS:
    print(f"\n{'='*60}")
    print(f"SuTraN (DA) — {cfg['log_name']}")
    print(f"{'='*60}")
    sutran_da.train_eval(log_name=cfg["log_name"])


SuTraN (DA) — Sepsis
18
device: cuda
Device: cuda
 
------------------------------------
EPOCH 0:
____________________________________


Batch calculation at epoch 0.: 66it [00:03, 19.73it/s]


End of epoch 0
Running average global loss: 0.26464380919933317 (over last 800 batches)
Running average activity prediction loss: 0.19530000269412995 (Cross Entropy over last 800 batches)
Running average time till next event prediction loss: 0.0233064048178494 (MAE over last 800 batches)
Running average (complete) remaining runtime prediction loss: 0.046037399880588054 (MAE over last 800 batches)


Validation batch calculation: 1it [00:01,  1.71s/it]


Avg MAE TTNE prediction validation set: 0.15077248215675354 (standardized) ; 5604.749479166667 (minutes)'
Avg 1-(normalized) DL distance acitivty suffix prediction validation set: 0.173039972782135
Percentage of suffixes predicted to END: too early - 0.14717348927875243 ; right moment - 0.03654970760233918 ; too late - 0.8162768031189084
Too early instances - avg amount of events too early: 10.069536209106445
Too late instances - avg amount of events too late: 31.637014389038086
Avg absolute amount of events predicted too early / too late: 27.306529998779297
Avg MAE RRT prediction validation set: 0.39684391021728516 (standardized) ; 41530.3 (minutes)'
 
------------------------------------
EPOCH 1:
____________________________________


...

End of epoch 110
Running average global loss: 0.14190223753452302 (over last 800 batches)
Running average activity prediction loss: 0.09082816779613495 (Cross Entropy over last 800 batches)
Running average time till next event prediction loss: 0.013474840931594371 (MAE over last 800 batches)
Running average (complete) remaining runtime prediction loss: 0.03759922880679369 (MAE over last 800 batches)


Validation batch calculation: 1it [00:01,  1.56s/it]


Avg MAE TTNE prediction validation set: 0.1339067816734314 (standardized) ; 5006.641145833333 (minutes)'
Avg 1-(normalized) DL distance acitivty suffix prediction validation set: 0.3425459563732147
Percentage of suffixes predicted to END: too early - 0.17543859649122806 ; right moment - 0.11062378167641325 ; too late - 0.7139376218323586
Too early instances - avg amount of events too early: 4.5722222328186035
Too late instances - avg amount of events too late: 27.383617401123047
Avg absolute amount of events predicted too early / too late: 20.352338790893555
Avg MAE RRT prediction validation set: 0.3867967426776886 (standardized) ; 40404.120833333334 (minutes)'
No improvements in validation loss for 24 consecutive epochs. Final epoch: 110
cuda


Validation batch calculation: 2it [00:02,  1.35s/it]


Avg MAE TTNE prediction validation set: 0.0798148363828659 (standardized) ; 2973.2877604166665 (minutes)'
Avg 1-(normalized) DL distance acitivty suffix prediction validation set: 0.3665546774864197
Percentage of suffixes predicted to END: too early - 0.14753623188405798 ; right moment - 0.12753623188405797 ; too late - 0.7249275362318841
Too early instances - avg amount of events too early: 3.8899803161621094
Too late instances - avg amount of events too late: 25.818471908569336
Avg absolute amount of events predicted too early / too late: 19.290433883666992
Avg MAE RRT prediction validation set: 0.21120361983776093 (standardized) ; 21958.25625 (minutes)'


---
## Model 2 — SuTraN (Non-Data-Aware)

**Approach from paper:** *SuTraN* (ICPM 2024) — NDA variant

Identical architecture to SuTraN (DA) above, but **restricted to control-flow information only**: each prefix event token consists solely of the activity label and the two numerical timestamp proxies (time since case start, time since previous event).  All additional categorical and numerical event/case attributes are discarded.

This variant is included to provide a fair comparison against the other NDA baselines (CRTP-LSTM NDA, ED-LSTM, SEP-LSTM, BEST) that also operate without data attributes.

In [4]:
import TRAIN_EVAL_SUTRAN_NDA as sutran_nda

for cfg in LOGS:
    print(f"\n{'='*60}")
    print(f"SuTraN (NDA) — {cfg['log_name']}")
    print(f"{'='*60}")
    sutran_nda.train_eval(log_name=cfg["log_name"], tss_index=cfg["tss_index"])


SuTraN (NDA) — Sepsis
18
device: cuda
Device: cuda
 
------------------------------------
EPOCH 0:
____________________________________


Batch calculation at epoch 0.: 66it [00:02, 31.88it/s]


End of epoch 0
Running average global loss: 0.26274111568927766 (over last 800 batches)
Running average activity prediction loss: 0.19601879209280015 (Cross Entropy over last 800 batches)
Running average time till next event prediction loss: 0.022435084944590927 (MAE over last 800 batches)
Running average (complete) remaining runtime prediction loss: 0.044287239592522384 (MAE over last 800 batches)


Validation batch calculation: 1it [00:01,  1.39s/it]


Avg MAE TTNE prediction validation set: 0.13963207602500916 (standardized) ; 5074.190625 (minutes)'
Avg 1-(normalized) DL distance acitivty suffix prediction validation set: 0.13258932530879974
Percentage of suffixes predicted to END: too early - 0.07504873294346978 ; right moment - 0.0014619883040935672 ; too late - 0.9234892787524367
Too early instances - avg amount of events too early: 12.396103858947754
Too late instances - avg amount of events too late: 31.291820526123047
Avg absolute amount of events predicted too early / too late: 29.827972412109375
Avg MAE RRT prediction validation set: 0.39569422602653503 (standardized) ; 41441.63333333333 (minutes)'
 
------------------------------------
EPOCH 1:
____________________________________


...

End of epoch 119
Running average global loss: 0.14569946110248566 (over last 800 batches)
Running average activity prediction loss: 0.09285154014825821 (Cross Entropy over last 800 batches)
Running average time till next event prediction loss: 0.013330807453021407 (MAE over last 800 batches)
Running average (complete) remaining runtime prediction loss: 0.03951711308211088 (MAE over last 800 batches)


Validation batch calculation: 1it [00:01,  1.40s/it]


Avg MAE TTNE prediction validation set: 0.13323289155960083 (standardized) ; 4997.440625 (minutes)'
Avg 1-(normalized) DL distance acitivty suffix prediction validation set: 0.22747136652469635
Percentage of suffixes predicted to END: too early - 0.04678362573099415 ; right moment - 0.03996101364522417 ; too late - 0.9132553606237817
Too early instances - avg amount of events too early: 4.885416507720947
Too late instances - avg amount of events too late: 30.367128372192383
Avg absolute amount of events predicted too early / too late: 27.96150016784668
Avg MAE RRT prediction validation set: 0.3922450840473175 (standardized) ; 41060.47083333333 (minutes)'
No improvements in validation loss for 24 consecutive epochs. Final epoch: 119
cuda


Validation batch calculation: 1it [00:02,  2.44s/it]


Avg MAE TTNE prediction validation set: 0.0784468874335289 (standardized) ; 2933.2408854166665 (minutes)'
Avg 1-(normalized) DL distance acitivty suffix prediction validation set: 0.23896551132202148
Percentage of suffixes predicted to END: too early - 0.035652173913043476 ; right moment - 0.05710144927536232 ; too late - 0.9072463768115943
Too early instances - avg amount of events too early: 4.056910514831543
Too late instances - avg amount of events too late: 30.521085739135742
Avg absolute amount of events predicted too early / too late: 27.83478355407715
Avg MAE RRT prediction validation set: 0.2153138667345047 (standardized) ; 22516.347916666666 (minutes)'


---
## Model 3 — CRTP-LSTM (Data-Aware)

**Approach from paper:** *Camargo, M., Dumas, M., González-Rojas, O. (2019). Learning Accurate LSTM Models of Business Processes. BPM 2019.*

CRTP-LSTM (Complete Remaining Trace Prediction LSTM) is a **multi-task LSTM** baseline:

- A **shared LSTM** encoder processes the prefix and feeds into multiple **dedicated LSTM** heads, one per prediction target.
- Prediction targets: activity label suffix, TTNE suffix, and RRT (remaining runtime).
- The suffix is generated by the LSTM in a single recurrent pass over the decoder input (teacher-forced during training, autoregressive at inference).
- Prefix events are **left-padded** so the final hidden state always corresponds to the last observed event.
- The DA variant uses all available event and case attributes.

Architecture defaults: `d_model=80`, 1 shared + 1 dedicated LSTM layer, dropout 0.2, NAdam optimiser with ReduceLROnPlateau, up to 500 epochs.

In [6]:
import TRAIN_EVAL_CRTP_LSTM_DA as crtp_da

for cfg in LOGS:
    print(f"\n{'='*60}")
    print(f"CRTP-LSTM (DA) — {cfg['log_name']}")
    print(f"{'='*60}")
    crtp_da.train_eval(log_name=cfg["log_name"])


CRTP-LSTM (DA) — Sepsis
18
device: cuda
Device: cuda
 
------------------------------------
EPOCH 0:
____________________________________


Batch calculation at epoch 0.: 66it [00:01, 44.64it/s]


End of epoch 0
Running average global loss: 0.10918722167611122 (over last 1600 batches)
Running average activity prediction loss: 0.08885611414909363 (Cross Entropy over last 1600 batches)
Running average (complete) remaining runtime prediction loss: 0.020331107396632433 (MAE over last 1600 batches)


Validation batch calculation: 2it [00:00,  2.79it/s]


Avg 1-(normalized) DL distance acitivty suffix prediction validation set: 0.14476251602172852
Percentage of suffixes predicted to END: too early - 0.01705653021442495 ; right moment - 0.0009746588693957114 ; too late - 0.9819688109161794
Too early instances - avg amount of events too early: 1.485714316368103
Too late instances - avg amount of events too late: 24.05459213256836
Avg absolute amount of events predicted too early / too late: 23.646198272705078
Avg MAE TTNE prediction validation set: 4950.451041666666 (minutes)'
Avg MAE RRT prediction validation set: 0.407967209815979 (standardized) ; 42729.0125 (minutes)'
Avg validation loss: 2.4217143058776855
 
------------------------------------
EPOCH 1:
____________________________________


...

End of epoch 62
Running average global loss: 0.05674198560416698 (over last 1600 batches)
Running average activity prediction loss: 0.04764545701444149 (Cross Entropy over last 1600 batches)
Running average (complete) remaining runtime prediction loss: 0.009096528394147754 (MAE over last 1600 batches)


Validation batch calculation: 2it [00:00,  2.62it/s]


Avg 1-(normalized) DL distance acitivty suffix prediction validation set: 0.46685874462127686
Percentage of suffixes predicted to END: too early - 0.44541910331384016 ; right moment - 0.1652046783625731 ; too late - 0.38937621832358676
Too early instances - avg amount of events too early: 6.083150863647461
Too late instances - avg amount of events too late: 5.291614532470703
Avg absolute amount of events predicted too early / too late: 4.769980430603027
Avg MAE TTNE prediction validation set: 6088.802604166666 (minutes)'
Avg MAE RRT prediction validation set: 0.44113466143608093 (standardized) ; 46167.88333333333 (minutes)'
Avg validation loss: 3.057382822036743
No improvements in validation loss for 59 consecutive epochs. Final epoch: 62
cuda


Validation batch calculation: 1it [00:00,  1.56it/s]

Avg 1-(normalized) DL distance acitivty suffix prediction validation set: 0.510138750076294
Percentage of suffixes predicted to END: too early - 0.5646376811594203 ; right moment - 0.20608695652173914 ; too late - 0.22927536231884058
Too early instances - avg amount of events too early: 5.140143871307373
Too late instances - avg amount of events too late: 4.026548862457275
Avg absolute amount of events predicted too early / too late: 3.825507164001465
Avg MAE TTNE prediction validation set: 4124.681510416666 (minutes)'
Avg MAE RRT prediction validation set: 0.2665189802646637 (standardized) ; 27591.3375 (minutes)'
Avg validation loss: 2.5315613746643066


---
## Model 4 — CRTP-LSTM (Non-Data-Aware)

**Approach from paper:** *Camargo et al. (2019)* — NDA variant

Same CRTP-LSTM architecture as Model 3, but restricted to the activity label and two timestamp proxies per event (no additional attributes).  Provides a fair NDA comparison partner to SuTraN NDA.

In [8]:
import TRAIN_EVAL_CRTP_LSTM_ND as crtp_nda

for cfg in LOGS:
    print(f"\n{'='*60}")
    print(f"CRTP-LSTM (NDA) — {cfg['log_name']}")
    print(f"{'='*60}")
    crtp_nda.train_eval(log_name=cfg["log_name"], tss_index=cfg["tss_index"])


CRTP-LSTM (NDA) — Sepsis
18
device: cuda
Device: cuda
 
------------------------------------
EPOCH 0:
____________________________________


Batch calculation at epoch 0.: 66it [00:00, 176.05it/s]


End of epoch 0
Running average global loss: 0.11045759797096252 (over last 1600 batches)
Running average activity prediction loss: 0.08958342835307122 (Cross Entropy over last 1600 batches)
Running average (complete) remaining runtime prediction loss: 0.02087416959926486 (MAE over last 1600 batches)


Validation batch calculation: 2it [00:00,  3.35it/s]


Avg 1-(normalized) DL distance acitivty suffix prediction validation set: 0.11308163404464722
Percentage of suffixes predicted to END: too early - 0.018518518518518517 ; right moment - 0.0 ; too late - 0.9814814814814815
Too early instances - avg amount of events too early: 1.9473683834075928
Too late instances - avg amount of events too late: 21.469711303710938
Avg absolute amount of events predicted too early / too late: 21.108186721801758
Avg MAE TTNE prediction validation set: 4963.4734375 (minutes)'
Avg MAE RRT prediction validation set: 0.5127159357070923 (standardized) ; 45737.53333333333 (minutes)'
Avg validation loss: 2.5868279933929443
 
------------------------------------
EPOCH 1:
____________________________________


...

End of epoch 79
Running average global loss: 0.07958968706429005 (over last 1600 batches)
Running average activity prediction loss: 0.061773586571216586 (Cross Entropy over last 1600 batches)
Running average (complete) remaining runtime prediction loss: 0.017816100604832172 (MAE over last 1600 batches)


Validation batch calculation: 2it [00:00,  3.29it/s]


Avg 1-(normalized) DL distance acitivty suffix prediction validation set: 0.3527092933654785
Percentage of suffixes predicted to END: too early - 0.2826510721247563 ; right moment - 0.0804093567251462 ; too late - 0.6369395711500975
Too early instances - avg amount of events too early: 5.791379451751709
Too late instances - avg amount of events too late: 12.16602897644043
Avg absolute amount of events predicted too early / too late: 9.385965347290039
Avg MAE TTNE prediction validation set: 5293.0875 (minutes)'
Avg MAE RRT prediction validation set: 0.4071321189403534 (standardized) ; 42624.041666666664 (minutes)'
Avg validation loss: 2.487135887145996
No improvements in validation loss for 59 consecutive epochs. Final epoch: 79
cuda


Validation batch calculation: 2it [00:00,  3.18it/s]

Avg 1-(normalized) DL distance acitivty suffix prediction validation set: 0.35387706756591797
Percentage of suffixes predicted to END: too early - 0.3652173913043478 ; right moment - 0.10782608695652174 ; too late - 0.5269565217391304
Too early instances - avg amount of events too early: 5.223809719085693
Too late instances - avg amount of events too late: 14.726072311401367
Avg absolute amount of events predicted too early / too late: 9.667825698852539
Avg MAE TTNE prediction validation set: 3116.675 (minutes)'
Avg MAE RRT prediction validation set: 0.21386782824993134 (standardized) ; 22396.747916666667 (minutes)'
Avg validation loss: 1.9930529594421387


---
## Model 5 — ED-LSTM (Encoder-Decoder LSTM)

**Approach based on:**  
- *Tax, N. et al. (2017). Predictive Business Process Monitoring with LSTM Neural Networks. CAiSE 2017.*  
- *Pfeiffer, P. et al. (2021). seq2seq Encoder-Decoder for Suffix Prediction. EDOC 2021.*

ED-LSTM is a **sequence-to-sequence encoder-decoder LSTM**:

- The **encoder LSTM** reads the full prefix and produces a fixed-size context vector (final hidden + cell state).
- The **decoder LSTM** is initialised with the encoder's final state and generates the activity and TTNE suffix step-by-step.
- Compared to the CRTP-LSTM, the decoder has its own recurrent state that is updated at each suffix step, which allows it to use previously predicted outputs as context.
- NDA only (activity label + timestamp proxies).

In [9]:
import TRAIN_EVAL_ED_LSTM as ed_lstm

for cfg in LOGS:
    print(f"\n{'='*60}")
    print(f"ED-LSTM — {cfg['log_name']}")
    print(f"{'='*60}")
    ed_lstm.train_eval(log_name=cfg["log_name"], tss_index=cfg["tss_index"])


ED-LSTM — Sepsis
18
device: cuda
Device: cuda
 
------------------------------------
EPOCH 0:
____________________________________


Batch calculation at epoch 0.: 66it [00:00, 103.75it/s]


End of epoch 0
Running average global loss: 0.24849363893270493 (over last 800 batches)
Running average activity prediction loss: 0.23260918289422988 (Cross Entropy over last 800 batches)
Running average time till next event prediction loss: 0.01588445599656552 (MAE over last 800 batches)


Validation batch calculation: 2it [00:00, 20.50it/s]


Avg MAE TTNE prediction validation set: 0.13371212780475616 (standardized) ; 4929.663020833334 (minutes)'
Avg 1-(normalized) DL distance acitivty suffix prediction validation set: 0.07971996068954468
Percentage of suffixes predicted to END: too early - 0.0 ; right moment - 0.0 ; too late - 1.0
Too early instances - avg amount of events too early: nan
Too late instances - avg amount of events too late: 31.69152069091797
Avg absolute amount of events predicted too early / too late: 31.69152069091797
Avg MAE RRT prediction validation set:  42432.379166666666 (minutes)'
 
------------------------------------
EPOCH 1:
____________________________________


...

End of epoch 199
Running average global loss: 0.1398961728811264 (over last 800 batches)
Running average activity prediction loss: 0.126536778062582 (Cross Entropy over last 800 batches)
Running average time till next event prediction loss: 0.013359394501894712 (MAE over last 800 batches)


Validation batch calculation: 2it [00:00,  8.00it/s]


Avg MAE TTNE prediction validation set: 0.13349494338035583 (standardized) ; 4983.424479166667 (minutes)'
Avg 1-(normalized) DL distance acitivty suffix prediction validation set: 0.16237396001815796
Percentage of suffixes predicted to END: too early - 0.032651072124756333 ; right moment - 0.024853801169590642 ; too late - 0.942495126705653
Too early instances - avg amount of events too early: 4.567163944244385
Too late instances - avg amount of events too late: 31.324716567993164
Avg absolute amount of events predicted too early / too late: 29.672513961791992
Avg MAE RRT prediction validation set:  52405.57083333333 (minutes)'
cuda


Validation batch calculation: 2it [00:00, 14.21it/s]


Avg MAE TTNE prediction validation set: 0.07879280298948288 (standardized) ; 2923.834895833333 (minutes)'
Avg 1-(normalized) DL distance acitivty suffix prediction validation set: 0.17627616226673126
Percentage of suffixes predicted to END: too early - 0.03536231884057971 ; right moment - 0.0408695652173913 ; too late - 0.923768115942029
Too early instances - avg amount of events too early: 3.754098415374756
Too late instances - avg amount of events too late: 31.609664916992188
Avg absolute amount of events predicted too early / too late: 29.332754135131836
Avg MAE RRT prediction validation set:  37035.67916666667 (minutes)'


---
## Model 6 — SEP-LSTM (Single-Event-Prediction LSTM)

**Approach from paper:** *Taymouri, F. et al. (2021). Predictive Business Process Monitoring via Generative Adversarial Nets. ICPM 2020 / BPM 2021.*

SEP-LSTM is a **one-step-ahead LSTM** repurposed for suffix generation through an **iterative external feedback loop**:

- The model is trained to predict only the **next** activity and TTNE given the full current prefix.
- At inference, the prediction for step $t$ is appended to the prefix as a new (synthesised) event token, and the model is queried again for step $t+1$.
- This loop continues until the END token is predicted or the maximum suffix length is reached.
- The new event token's timestamp features (time-since-start, time-since-previous) are computed from the predicted TTNE.
- NDA only; uses left-padded prefix tensors.

In [ ]:
import TRAIN_EVAL_SEP_LSTM as sep_lstm

for cfg in LOGS:
    print(f"\n{'='*60}")
    print(f"SEP-LSTM — {cfg['log_name']}")
    print(f"{'='*60}")
    sep_lstm.train_eval(log_name=cfg["log_name"], tss_index=cfg["tss_index"])


SEP-LSTM — Sepsis
18
device: cuda


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/rnn.py:82: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


Device: cuda
 
------------------------------------
EPOCH 0:
____________________________________


Batch calculation at epoch 0.: 66it [00:00, 96.60it/s] 


End of epoch 0
Running average global loss: 0.10285017408430576 (over last 1600 batches)
Running average activity prediction loss: 0.07727258935570717 (Cross Entropy over last 1600 batches)
Running average time till next event prediction loss: 0.025577584486454724 (MAE over last 1600 batches)


...

Batch calculation at epoch 26.: 66it [00:00, 121.63it/s]


End of epoch 26
Running average global loss: 0.0413223647326231 (over last 1600 batches)
Running average activity prediction loss: 0.03745229013264179 (Cross Entropy over last 1600 batches)
Running average time till next event prediction loss: 0.00387007457902655 (MAE over last 1600 batches)


Validation batch calculation: 1it [00:00,  7.13it/s]


Avg validation loss: 1.0439770221710205
Avg MAE TTNE Standardized: 0.07228674739599228
Avg MAE TTNE minutes: 2722.000667313799
Avg Accuracy Next activity prediction 0.6271929740905762
Avg CE validation set: 0.97169029712677
 
------------------------------------
EPOCH 27:
____________________________________


Batch calculation at epoch 27.: 37it [00:00, 119.39it/s]

---
## Model 7 — BEST (Bilaterally Expanding Subtrace Tree)

**Approach from paper:** *Rauch, S., Frey, C. M. M., Maldonado, A. J., & Seidl, T. (2025). BEST: Bilaterally Expanding Subtrace Tree for Event Sequence Prediction. BPM 2025. Springer, LNCS 16044.*  
🏆 Runner-up Best Student Paper Award, BPM 2025

BEST is a **non-parametric, data-mining-based** baseline — it requires **no gradient-based optimisation** at all:

- **Fitting:** For every training instance the complete trace (prefix + suffix) is reconstructed. All sub-sequences of length 1 to `max_context_length` (default 10) are extracted and stored as conditional next-activity frequency counts in a dictionary-based n-gram table.
- **Prediction:** Given a test prefix, BEST performs a **longest-context-first back-off search**: it tries to match the last $k$ activities in the current sequence against the table (starting at $k=10$, backing off to $k=1$), and picks the activity with the highest conditional count. Falls back to the global unigram mode if no match is found.
- **Suffix generation:** Activities are generated autoregressively until the END token is predicted.
- **Control-flow only (NDA):** No timestamp or attribute information is used.
- **Time metrics:** Since BEST does not predict timestamps, TTNE and RRT are approximated with a constant predictor equal to the training mean TTNE.

Despite its simplicity, BEST achieves competitive or superior activity-suffix DL similarity compared to deep learning models on several benchmark logs.

In [4]:
import TRAIN_EVAL_BEST as best

for cfg in LOGS:
    print(f"\n{'='*60}")
    print(f"BEST — {cfg['log_name']}")
    print(f"{'='*60}")
    best.train_eval(log_name=cfg["log_name"])


BEST — Sepsis
num_activities: 18
Fitting BEST model on training data ...


Building BEST pattern tree: 100%|████████████████████| 8353/8353 [00:00<00:00, 21110.75it/s]


BEST model fitted successfully.
BEST model saved to: results_per_log/Sepsis/BEST_results/best_model.pkl


Computing DL similarity: 100%|████████████████████████████████| 7/7 [00:02<00:00,  2.87it/s]


=== BEST Test Set Results ===
Avg 1-(normalised) DL similarity activity suffix: 0.43147921562194824
Percentage of suffixes predicted to END: too early - 0.20057971014492754 ; right moment - 0.07420289855072464 ; too late - 0.7252173913043478
Too early instances -- avg # events too early: 5.937861442565918
Too late  instances -- avg # events too late:  10.298161506652832
Avg absolute length difference: 8.659420013427734
Avg MAE TTNE (constant-mean predictor): 6346.486458333334 (minutes)
Avg MAE RRT  (constant-mean predictor): 74402.96666666666 (minutes)


---
# Part 3 — Results Summary

After all cells above have finished, each model's results are stored as pickle files under:

```
<log_name>/
  SUTRAN_DA_results/TEST_SET_RESULTS/averaged_results.pkl
  SUTRAN_NDA_results/TEST_SET_RESULTS/averaged_results.pkl
  CRTP_LSTM_DA_results/TEST_SET_RESULTS/averaged_results.pkl
  CRTP_LSTM_ND_results/TEST_SET_RESULTS/averaged_results.pkl
  ED_LSTM_results/TEST_SET_RESULTS/averaged_results.pkl
  SEP_LSTM_results/TEST_SET_RESULTS/averaged_results.pkl
  BEST_results/TEST_SET_RESULTS/averaged_results.pkl
```

Each `averaged_results.pkl` is a dict with keys `"DL sim"`, `"MAE TTNE minutes"`, `"MAE RRT minutes"`.

The cell below collects and displays them in a summary table.

In [10]:
import os
import pickle
import pandas as pd

RESULT_DIRS = {
    "SuTraN (DA)":       "SUTRAN_DA_results",
    "SuTraN (NDA)":      "SUTRAN_NDA_results",
    "CRTP-LSTM (DA)":    "CRTP_LSTM_DA_results",
    "CRTP-LSTM (NDA)":   "CRTP_LSTM_NDA_results",
    "ED-LSTM":           "ED_LSTM_results",
    "SEP-LSTM":          "SEP_LSTM_results",
    "BEST":              "BEST_results",
}

rows = []
for cfg in LOGS:
    log = cfg["log_name"]
    for model_name, result_dir in RESULT_DIRS.items():
        path = os.path.join("results_per_log",log, result_dir, "TEST_SET_RESULTS", "averaged_results.pkl")
        if os.path.exists(path):
            with open(path, "rb") as f:
                res = pickle.load(f)
            rows.append({
                "Log":               log,
                "Model":             model_name,
                "DL similarity ↑":   round(res.get("DL sim", float("nan")), 4),
                "MAE TTNE (min) ↓":  round(res.get("MAE TTNE minutes", float("nan")), 2),
                "MAE RRT (min) ↓":   round(res.get("MAE RRT minutes", float("nan")), 2),
            })
        else:
            rows.append({
                "Log": log, "Model": model_name,
                "DL similarity ↑": None,
                "MAE TTNE (min) ↓": None,
                "MAE RRT (min) ↓": None,
            })

df_results = pd.DataFrame(rows).set_index(["Log", "Model"])
pd.set_option("display.float_format", "{:.4f}".format)
display(df_results)

DL similarity ↑  MAE TTNE (min) ↓  MAE RRT (min) ↓
Log    Model                                                              
Sepsis SuTraN (DA)               0.3666         2973.2900       21958.2600
       SuTraN (NDA)              0.2390         2933.2400       22516.3500
       CRTP-LSTM (DA)            0.5101         4124.6800       27591.3400
       CRTP-LSTM (NDA)           0.3539         3116.6800       22396.7500
       ED-LSTM                   0.1763         2923.8300       37035.6800
       SEP-LSTM                  0.3923         2886.1900       25338.2600
       BEST                      0.4315         6346.4900       74402.9700